# 🎬 Movie Recommendation System

## Project Overview
This project builds a **Content-Based Movie Recommendation System** using the TMDB 5000 Movie Dataset.
Given a movie title, the system recommends the top 5 most similar movies based on metadata such as
genres, keywords, cast, crew, and plot overview.

---

## Objectives
1. Perform exploratory data analysis (EDA) on the TMDB dataset.
2. Clean and preprocess raw movie metadata.
3. Engineer a combined `tags` feature for each movie.
4. Build a content-based recommendation engine using **Bag of Words + Cosine Similarity**.
5. Visualise key insights from the dataset.
6. Evaluate the model qualitatively and discuss limitations.

---

## Dataset
| File | Description |
|------|-------------|
| `tmdb_5000_movies.csv` | 4803 movies with budget, genres, keywords, overview, ratings, etc. |
| `tmdb_5000_credits.csv` | Cast and crew information for each movie |

---

## Methodology
```
Raw Data  →  EDA  →  Preprocessing  →  Feature Engineering  →  Vectorisation  →  Cosine Similarity  →  Recommendations
```

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import ast
import nltk
import pickle
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.porter import PorterStemmer

try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    WORDCLOUD_AVAILABLE = False
    print('wordcloud not installed – skipping word-cloud cell')

warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 2. Load Data

In [ ]:
movies  = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

print(f'Movies  dataset: {movies.shape[0]:,} rows × {movies.shape[1]} columns')
print(f'Credits dataset: {credits.shape[0]:,} rows × {credits.shape[1]} columns')

In [ ]:
movies.head(3)

In [ ]:
credits.head(3)

## 3. Exploratory Data Analysis

### 3.1 Shape & Basic Info

In [ ]:
print('=== Movies Dataset ===')
print(f'Shape: {movies.shape}')
movies.info()
print()
print('=== Credits Dataset ===')
print(f'Shape: {credits.shape}')
credits.info()

### 3.2 Summary Statistics

In [ ]:
movies.describe()

### 3.3 Missing Value Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Bar chart of missing values ──
missing = movies.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
axes[0].barh(missing.index, missing.values, color='salmon')
axes[0].set_xlabel('Missing Count')
axes[0].set_title('Missing Values – Movies Dataset')
for bar, val in zip(axes[0].patches, missing.values):
    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                 str(val), va='center')

# ── Heatmap of nulls (first 200 rows for readability) ──
sns.heatmap(movies.iloc[:200].isnull(), yticklabels=False,
            cbar=False, cmap='viridis', ax=axes[1])
axes[1].set_title('Null Heatmap (first 200 rows) – Movies Dataset')

plt.tight_layout()
plt.show()

print('\nMissing values per column (movies):')
print(movies.isnull().sum()[movies.isnull().sum() > 0])
print('\nMissing values per column (credits):')
print(credits.isnull().sum()[credits.isnull().sum() > 0])

### 3.4 Duplicate Detection

In [ ]:
print(f'Duplicate rows in movies  dataset: {movies.duplicated().sum()}')
print(f'Duplicate rows in credits dataset: {credits.duplicated().sum()}')

### 3.5 Key Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Vote average distribution ──
axes[0].hist(movies['vote_average'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Vote Average')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Movie Ratings (vote_average)')

# ── Top 10 most popular movies ──
top10 = movies.nlargest(10, 'popularity')[['title', 'popularity']]
axes[1].barh(top10['title'], top10['popularity'], color='teal')
axes[1].invert_yaxis()
axes[1].set_xlabel('Popularity Score')
axes[1].set_title('Top 10 Most Popular Movies')

plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Feature Engineering

### 4.1 Merge Datasets

In [ ]:
# Merge on title; also join on movie_id to disambiguate any title duplicates
movies = movies.merge(credits, on='title')

# Validate merge result
print(f'Merged shape: {movies.shape}')
if movies.duplicated(subset='movie_id').sum() > 0:
    print(f'Dropping {movies.duplicated(subset="movie_id").sum()} duplicate movie_id rows from merge...')
    movies.drop_duplicates(subset='movie_id', inplace=True)
    print(f'Shape after deduplication: {movies.shape}')

movies.head(2)

### 4.2 Select Relevant Columns

In [ ]:
movies = movies[['movie_id', 'title', 'overview', 'genres',
                  'keywords', 'cast', 'crew', 'vote_average', 'popularity']]
movies.head(2)

### 4.3 Handle Missing Values

In [ ]:
print(f'Rows before dropping nulls: {len(movies)}')
movies.dropna(inplace=True)
print(f'Rows after  dropping nulls: {len(movies)}')

### 4.4 Parse JSON-Like String Columns

In [ ]:
def convert(obj: str) -> list:
    """Parse a JSON-like string and return a list of 'name' values.

    Parameters
    ----------
    obj : str
        A JSON-formatted string containing a list of dicts with a 'name' key.

    Returns
    -------
    list
        List of name strings extracted from the parsed object.
    """
    return [item['name'] for item in ast.literal_eval(obj)]


def convert3(obj: str) -> list:
    """Parse a JSON-like cast string and return the top 3 actor names.

    Parameters
    ----------
    obj : str
        A JSON-formatted string for the cast column.

    Returns
    -------
    list
        Up to three actor name strings.
    """
    return [item['name'] for item in ast.literal_eval(obj)[:3]]


def fetch_director(obj: str) -> list:
    """Extract the director name(s) from a crew JSON-like string.

    Parameters
    ----------
    obj : str
        A JSON-formatted string for the crew column.

    Returns
    -------
    list
        A list containing the director's name (empty if not found).
    """
    return [item['name'] for item in ast.literal_eval(obj)
            if item['job'] == 'Director']


# Apply parsing functions
movies['genres']   = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast']     = movies['cast'].apply(convert3)
movies['crew']     = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

movies.head(3)

### 4.5 Genre Distribution (After Parsing)

In [ ]:
from collections import Counter

# Flatten all genre lists and count occurrences
all_genres = [genre for sublist in movies['genres'] for genre in sublist]
genre_counts = Counter(all_genres).most_common(15)
genres_df = pd.DataFrame(genre_counts, columns=['Genre', 'Count'])

plt.figure(figsize=(12, 6))
sns.barplot(data=genres_df, x='Count', y='Genre', palette='viridis')
plt.title('Top 15 Most Common Genres')
plt.xlabel('Number of Movies')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()

### 4.6 Clean Text – Remove Spaces to Avoid Ambiguity

In [ ]:
def collapse(tokens: list) -> list:
    """Remove spaces within each token to prevent word merging issues.

    For example 'Sam Worthington' becomes 'samworthington' so that
    'Sam' alone is not treated as a separate feature.

    Parameters
    ----------
    tokens : list
        List of string tokens.

    Returns
    -------
    list
        List of lowercase, space-free tokens.
    """
    return [token.replace(' ', '').lower() for token in tokens]


movies['genres']   = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)
movies['cast']     = movies['cast'].apply(collapse)
movies['crew']     = movies['crew'].apply(collapse)

movies.head(3)

### 4.7 Create `tags` Column

In [ ]:
# Combine all textual features into a single 'tags' string
movies['tags'] = (movies['overview'] + movies['genres'] +
                  movies['keywords'] + movies['cast'] + movies['crew'])

# Keep only the columns needed for the recommendation engine
new_df = movies[['movie_id', 'title', 'tags', 'vote_average', 'popularity']].copy()

# Convert list of tokens to a single lowercased string
new_df['tags'] = new_df['tags'].apply(lambda x: ' '.join(x).lower())

print('Sample tags for the first movie:')
print(new_df['tags'].iloc[0])

### 4.8 Stemming

In [ ]:
ps = PorterStemmer()


def stem(text: str) -> str:
    """Apply Porter stemming to each word in a text string.

    Stemming reduces inflected words to their root form
    (e.g. 'dancing' → 'danc', 'loved' → 'love').

    Parameters
    ----------
    text : str
        The input text to stem.

    Returns
    -------
    str
        Space-joined string of stemmed words.
    """
    return ' '.join(ps.stem(word) for word in text.split())


new_df['tags'] = new_df['tags'].apply(stem)

print('Sample tags after stemming:')
print(new_df['tags'].iloc[0])

## 5. Visualisations

### 5.1 Word Cloud of Tags

In [ ]:
if WORDCLOUD_AVAILABLE:
    all_tags = ' '.join(new_df['tags'].tolist())
    wc = WordCloud(width=900, height=450, background_color='white',
                   max_words=200, colormap='plasma').generate(all_tags)
    plt.figure(figsize=(14, 7))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud of Movie Tags', fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print('Install wordcloud (`pip install wordcloud`) to see this visualisation.')

## 6. Content-Based Filtering

We use a **Bag of Words** model:
1. Each movie's `tags` string is vectorised with `CountVectorizer` (top 5000 features, English stop words removed).
2. **Cosine similarity** between all pairs of movie vectors is computed.
3. For any query movie, we rank all other movies by similarity score and return the top 5.

### 6.1 Vectorise Tags

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

print(f'Vocabulary size : {len(cv.get_feature_names_out())}')
print(f'Vector matrix   : {vectors.shape}')

### 6.2 Compute Cosine Similarity

In [ ]:
similarity = cosine_similarity(vectors)
print(f'Similarity matrix shape: {similarity.shape}')

### 6.3 Recommendation Function

In [ ]:
def recommend(movie: str, top_n: int = 5) -> list:
    """Return the top-N most similar movies to the given title.

    Uses pre-computed cosine similarity scores over the bag-of-words
    representation of each movie's combined metadata tags.

    Parameters
    ----------
    movie : str
        Exact title of the query movie (case-sensitive).
    top_n : int, optional
        Number of recommendations to return (default 5).

    Returns
    -------
    list
        List of recommended movie title strings.

    Raises
    ------
    ValueError
        If the movie title is not found in the dataset.
    """
    # Locate the movie in the dataset
    matches = new_df[new_df['title'] == movie]
    if matches.empty:
        raise ValueError(f"Movie '{movie}' not found in dataset.")

    movie_index = matches.index[0]

    # Get similarity scores for this movie against all others
    distances = similarity[movie_index]

    # Sort descending; skip index 0 (the movie itself)
    ranked = sorted(enumerate(distances), key=lambda x: x[1], reverse=True)[1: top_n + 1]

    return [new_df.iloc[i].title for i, _ in ranked]

## 7. Example Recommendations

In [ ]:
def print_recommendations(title: str) -> None:
    """Pretty-print the top-5 recommendations for a given movie."""
    print(f"\n{'─' * 50}")
    print(f"  🎬  Because you watched: {title}")
    print(f"{'─' * 50}")
    try:
        recs = recommend(title)
        for rank, rec in enumerate(recs, start=1):
            print(f"  {rank}. {rec}")
    except ValueError as err:
        print(f"  ⚠️  {err}")


for query in ['Avatar', 'The Dark Knight Rises', 'Spectre']:
    print_recommendations(query)

## 8. Save Model Artefacts

In [ ]:
pickle.dump(new_df, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
print('Saved movies.pkl and similarity.pkl')

## 9. Evaluation & Observations

### Strengths
| Property | Detail |
|----------|--------|
| No user data required | Works purely on movie metadata – no need for rating history |
| Fast inference | Cosine similarity lookup is O(n) once the matrix is built |
| Interpretable | Features (genres, cast, keywords) are human-readable |

### Limitations
| Limitation | Description |
|------------|-------------|
| **Cold-start (new movies)** | New movies with no metadata cannot be recommended |
| **Popularity bias** | The model ignores user preferences and ratings |
| **Vocabulary mismatch** | Bag-of-words loses word order and semantic meaning |
| **Static similarity** | The matrix must be recomputed when new movies are added |

### Possible Improvements
- **Collaborative Filtering** – leverage user–item interaction matrices (e.g. SVD, ALS)
- **Hybrid Approach** – combine content-based scores with collaborative signals
- **TF-IDF / Word Embeddings** – replace Bag-of-Words with TF-IDF or word2vec for richer semantics
- **Deep Learning** – use a neural network to learn latent item representations
- **Real-time updates** – use approximate nearest-neighbour search (e.g. FAISS) for scalable, live recommendations


## 10. Conclusion

This notebook demonstrated an end-to-end content-based movie recommendation system:

1. **Data Loading** – imported TMDB movies and credits datasets.
2. **EDA** – explored shapes, data types, missing values, and key distributions.
3. **Preprocessing** – merged datasets, parsed JSON-like columns, dropped nulls.
4. **Feature Engineering** – constructed a unified `tags` column and applied Porter stemming.
5. **Modelling** – vectorised tags with `CountVectorizer` and computed pairwise cosine similarity.
6. **Recommendations** – demonstrated qualitative results for *Avatar*, *The Dark Knight Rises*, and *Spectre*.
7. **Persistence** – serialised the model artefacts (`movies.pkl`, `similarity.pkl`) for use in the Flask app.

The system can be extended with collaborative filtering or deep-learning approaches to handle personalisation
and cold-start challenges at scale.
